# 06 梯度裁剪：梯度太大时先缩小

## 知识点标注
- 行业状态：常见工程实践。
- 现实用途：防止某一次更新过大，导致 loss 爆炸或训练不稳定。
- 本课类型：本地概念演示，不使用 GPU，不训练 MokioMind。

本课只研究一个动作：`clip_grad_norm_` 如何改变梯度大小，以及它是否改变参数。先记住核心规则：如果整体梯度范数超过阈值，就把所有梯度乘上同一个缩放比例。

## 先区分三件事

```text
backward()              计算梯度
clip_grad_norm_(...)    必要时缩小梯度
optimizer.step()        使用梯度修改参数
```

梯度裁剪不是重新计算梯度，也不是直接修改参数。

In [1]:
import torch

w = torch.tensor([1.0], requires_grad=True)
loss = (w - 5.0) ** 2
loss.backward()
print('backward 后：w =', w.item(), 'grad =', w.grad.item())

before = w.item()
old_norm = torch.nn.utils.clip_grad_norm_([w], max_norm=1.0)
print('裁剪前梯度范数：', old_norm.item())
print('裁剪后：w =', w.item(), 'grad =', w.grad.item())

backward 后：w = 1.0 grad = -8.0
裁剪前梯度范数： 8.0
裁剪后：w = 1.0 grad = -0.9999998807907104


### 先预测

`loss=(w-5)^2` 且 `w=1` 时，梯度是 `-8`。如果 `max_norm=1.0`：

1. 裁剪前 `w` 会不会改变？
2. 裁剪后梯度大约是多少？
3. 这一步会不会让梯度方向反过来？

对于只有一个参数的例子：

```text
原梯度 g = -8
原范数 ||g|| = 8
阈值 max_norm = 1
缩放比例 = 1 / 8 = 0.125
新梯度 = -8 × 0.125 = -1
```

因为缩放比例是正数，所以只改变大小，不改变正负方向。

In [2]:
optimizer = torch.optim.SGD([w], lr=0.1)
print('step 前：w =', w.item(), 'grad =', w.grad.item())
optimizer.step()
print('step 后：w =', w.item())
optimizer.zero_grad()
print('zero_grad 后：w.grad =', w.grad)

step 前：w = 1.0 grad = -0.9999998807907104
step 后：w = 1.100000023841858
zero_grad 后：w.grad = None


## 对应 MokioMind 真实代码

项目中与本课对应的最小主线是：

```python
scaler.scale(loss).backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
scaler.step(optimizer)
optimizer.zero_grad(set_to_none=True)
```

逐行理解：

1. `backward()` 把每个模型参数的梯度写入 `.grad`。
2. `clip_grad_norm_` 检查所有参数梯度的整体范数；超过阈值时按比例缩小。
3. `scaler.step(optimizer)` 读取处理后的梯度，真正修改参数。
4. `zero_grad(...)` 清空旧梯度，为下一次累积准备。

真实位置：[trainer/train_pretrain.py:65-79](/Users/mrwong/Documents/Program/AI/Minimind/eg/MokioMind-master/trainer/train_pretrain.py:65)。阈值默认值在 [trainer/train_pretrain.py:165](/Users/mrwong/Documents/Program/AI/Minimind/eg/MokioMind-master/trainer/train_pretrain.py:165)：`grad_clip=1.0`。项目中的 `scaler.step(optimizer)` 才是参数更新；它是混合精度训练对 `optimizer.step()` 的包装。

## 验收题

1. `clip_grad_norm_` 修改的是参数 `w`、梯度 `w.grad`，还是两者都不修改？
2. 如果原梯度方向是负的，裁剪后方向会变成正的吗？
3. 为什么项目把梯度裁剪放在 `backward()` 后、`step()` 前？
4. 它和 `accumulation_steps` 的职责分别是什么？